# PoC MIA104 — cliente de la API Python

Este notebook no reimplementa el workflow ni es el pipeline de datos de la Clase 7. El núcleo de procesamiento está en `src.pipeline` y funciona también sin Jupyter.

**Requisitos:** instalar las dependencias del repositorio con Python 3.12, FFmpeg, modelos y autorizaciones correspondientes; usar solamente audio permitido para pruebas.


## 1. Importar la API

Ejecutar Jupyter desde la raíz del repositorio (o instalar el paquete con `pip install -e .`).

In [ ]:
from dataclasses import replace
from pathlib import Path

from src.rutas import DIRECTORIO_GRABACIONES_PREDETERMINADO, DIRECTORIO_RESULTADOS_PREDETERMINADO
from src.minutas.generacion import ConfiguracionMinutas
from src.pipeline import (
    SolicitudReunion, preparar_reunion, continuar_reunion,
    procesar_reunion, consultar_reunion, reetiquetar_reunion, leer_resultado_texto,
)


## 2. Configurar una reunión

Modificar las rutas y modelos según el equipo. El orden de las fuentes será el orden de concatenación. **No usar audios privados ni credenciales en el notebook versionado.** `permitir_remoto=False` impide transferencias a proveedores externos; si se habilita, debe existir autorización consciente de quien aporta los datos.

In [ ]:
fuentes = (DIRECTORIO_GRABACIONES_PREDETERMINADO / "audio_reunion_01.wav",)  # reemplazar por una o varias rutas existentes
salida = DIRECTORIO_RESULTADOS_PREDETERMINADO / "reunion-ejemplo"

config_minutas = ConfiguracionMinutas(
    modelo="qwen2.5-7b",
    secciones=("resumen", "temas", "decisiones", "acciones"),
    permitir_remoto=False,
)
solicitud = SolicitudReunion(
    fuentes=fuentes,
    salida=salida,
    modelo_stt="medium",
    dispositivo_diarizacion="auto",
    identificacion="omitir",
    minutas=config_minutas,
)


## 3. Ejecutar de punta a punta

La operación de alto nivel ejecuta ingesta, diarización y STT sobre el mismo WAV, asociación temporal y minutas; las etapas con artefactos válidos pueden reutilizarse en corridas posteriores. Ejecutar solamente después de indicar fuentes reales accesibles.

In [ ]:
if not all(Path(ruta).expanduser().is_file() for ruta in solicitud.fuentes):
    raise FileNotFoundError("Reemplazá fuentes con audios existentes antes de ejecutar.")

resultado = procesar_reunion(solicitud)
print(resultado["estado"], resultado.get("error"))
print(*resultado["artefactos"].keys(), sep="\n - ")


## 4. Consultar resultados y trazabilidad

La lectura del estado no ejecuta modelos; informa solamente los artefactos vigentes. El texto de la minuta se obtiene mediante una lectura verificada bajo bloqueo, como en la interfaz.

In [ ]:
consulta = consultar_reunion(salida)
print(consulta["estado"])
if "minutes.md" in consulta["artefactos"]:
    print(leer_resultado_texto(salida, "minutes.md"))


## 5. Identificación opcional desde Python

Para evaluación modular, puede separarse la preparación de la publicación final. La GUI utiliza las fichas existentes; Python puede continuar sin nombres o aportar nombres con `reetiquetar_reunion()`.

El bloque siguiente es ilustrativo y no se ejecuta automáticamente.

In [ ]:
# solicitud_con_pausa = replace(solicitud, identificacion="pausa")
# preparar_reunion(solicitud_con_pausa)
# continuar_reunion(salida)                  # publicar sin identificación
# reetiquetar_reunion(salida, {"SPEAKER_00": "Ana"})  # no repetir inferencias vigentes


## Alcance de la validación

Las pruebas automáticas del workflow verifican contratos, fallos y reanudación con trabajadores simulados; este notebook permite la ejecución real cuando se dispone de modelos, servicios y hardware compatibles. No sustituye la evaluación del Hito 4 ni demuestra equivalencia computacional entre modelos locales y remotos.
